In [104]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
import talib
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
import torch.optim as optim
import os
from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau 
import shap
import plotly.graph_objs as go
import plotly.offline as pyo
from tqdm.auto import tqdm


In [105]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("gpu")
else:
    device = torch.device('cpu')
print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
print('cuDNN version:', torch.backends.cudnn.version())

gpu
2.1.2+cu121
CUDA available: True
CUDA version: 12.1
cuDNN version: 8902


In [106]:
start_date = '2018-01-01'
end_date = '2024-01-24'

stock_data = yf.download("AAPL", start=start_date, end=end_date)[["Adj Close", "High", "Low", "Volume"]]

stock_data = stock_data.reset_index()

stock_data = stock_data[['Date', 'Adj Close', "High", "Low", "Volume"]]

stock_data = stock_data.sort_values(by="Date")
stock_data.head(45)

[*********************100%%**********************]  1 of 1 completed


,Date,Adj Close,High,Low,Volume
0,2018-01-02,40.722874,43.075001,42.314999,102223600
1,2018-01-03,40.715771,43.637501,42.990002,118071600
2,2018-01-04,40.904900,43.367500,43.020000,89738400
3,2018-01-05,41.370628,43.842499,43.262501,94640000
4,2018-01-08,41.216972,43.902500,43.482498,82271200
5,2018-01-09,41.212234,43.764999,43.352501,86336000
6,2018-01-10,41.202774,43.575001,43.250000,95839600
7,2018-01-11,41.436817,43.872501,43.622501,74670800
8,2018-01-12,41.864704,44.340000,43.912498,101672400
9,2018-01-16,41.651939,44.847500,44.035000,118263600


In [107]:
time_step = 44

In [108]:
test_index = int((len(stock_data)-44)*0.8+44+44)

In [109]:
date = stock_data["Date"].iloc[time_step:].dt.strftime('%Y-%m-%d')
date_test = stock_data["Date"].iloc[test_index:].reset_index()
date_test.drop(columns=["index"], inplace=True)
date_test

,Date
0,2023-01-23
1,2023-01-24
2,2023-01-25
3,2023-01-26
4,2023-01-27
...,...
247,2024-01-17
248,2024-01-18
249,2024-01-19
250,2024-01-22


In [110]:
def add_technical_indicators(data, timeperiod=time_step):

    # MACD
    macd, macdsignal, macdhist = talib.MACD(data["Adj Close"], fastperiod=12, slowperiod=26, signalperiod=9)

    rsi = talib.RSI(data["Adj Close"], timeperiod=14)

    # CMO
    cmo = talib.CMO(data["Adj Close"], timeperiod=timeperiod)

    # MOM
    mom = talib.MOM(data["Adj Close"], timeperiod=timeperiod)

    # Bollinger Bands
    upperband, middleband, lowerband = talib.BBANDS(data["Adj Close"], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)

    # SMA
    sma = talib.SMA(data["Adj Close"], timeperiod=timeperiod)

    # Calculate Exponential Moving Average (EMA)
    ema = talib.EMA(data["Adj Close"], timeperiod=timeperiod)

    # Calculate Stochastic Oscillator
    slowk, slowd = talib.STOCH(data['High'], data['Low'], data['Adj Close'], fastk_period=14, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)

    # Calculate Average True Range (ATR)
    atr = talib.ATR(data['High'], data['Low'], data['Adj Close'], timeperiod=14)

    # Calculate On-Balance Volume (OBV)
    obv = talib.OBV(data['Adj Close'], data['Volume'])

    # Combine all indicators into a DataFrame
    indicators = pd.DataFrame({
        'MACD': macd,
        'MACD_signal': macdsignal,
        'RSI': rsi,
        'CMO': cmo,
        'MOM': mom,
        'Upper_BB': upperband,
        'Middle_BB': middleband,
        'Lower_BB': lowerband,
        'SMA': sma,
        'EMA': ema,
        'SLOWK': slowk,
        'SLOWD': slowd,
        'ATR': atr,
        'OBV': obv,

    })
    return indicators

In [111]:
indicators = add_technical_indicators(stock_data)
indicators.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0


In [112]:
indicators_with_price = pd.concat([indicators, stock_data["Adj Close"]], axis=1, join='inner')
indicators_with_price.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0,40.722874
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0,40.715771
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0,40.904900
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0,41.370628
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0,41.216972
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0,41.212234
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0,41.202774
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0,41.436817
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0,41.864704
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0,41.651939


In [113]:
indicators_with_price = indicators_with_price.dropna()
indicators_with_price = indicators_with_price.reset_index(drop=True)
indicators_with_price

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,0.420517,0.238064,56.052912,4.051362,0.823540,44.028288,40.539885,37.051483,40.615036,40.638546,12.020059,29.597806,2.704888,-4.208332e+08,41.546413
1,0.431503,0.276752,59.233517,6.192273,1.284019,44.042948,40.754083,37.465218,40.644218,40.699045,-9.379590,12.287804,2.706938,-3.257368e+08,41.999790
2,0.492754,0.319952,63.732478,9.481629,1.816479,43.867406,41.056252,38.245097,40.685502,40.788927,-18.948467,-5.435999,2.727886,-1.969960e+08,42.721378
3,0.568075,0.369577,66.042461,11.303189,1.763767,43.662505,41.356640,39.050774,40.725588,40.893170,-6.400702,-11.576253,2.738474,-6.816760e+07,43.134396
4,0.587478,0.413157,61.780488,9.044897,1.502037,43.567673,41.561488,39.555302,40.759725,40.974318,1.684858,-7.888104,2.738627,-1.949416e+08,42.719009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,190.372500,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993
1476,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,190.459545,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005
1477,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,190.553182,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998
1478,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,190.686818,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999


In [114]:
indicators_with_price['Next_Adj_Close'] = indicators_with_price['Adj Close'].shift(-1)
indicators_with_price['Signal'] = (indicators_with_price['Next_Adj_Close'] > indicators_with_price['Adj Close']).astype(int)
indicators_with_price


,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Next_Adj_Close,Signal
0,0.420517,0.238064,56.052912,4.051362,0.823540,44.028288,40.539885,37.051483,40.615036,40.638546,12.020059,29.597806,2.704888,-4.208332e+08,41.546413,41.999790,1
1,0.431503,0.276752,59.233517,6.192273,1.284019,44.042948,40.754083,37.465218,40.644218,40.699045,-9.379590,12.287804,2.706938,-3.257368e+08,41.999790,42.721378,1
2,0.492754,0.319952,63.732478,9.481629,1.816479,43.867406,41.056252,38.245097,40.685502,40.788927,-18.948467,-5.435999,2.727886,-1.969960e+08,42.721378,43.134396,1
3,0.568075,0.369577,66.042461,11.303189,1.763767,43.662505,41.356640,39.050774,40.725588,40.893170,-6.400702,-11.576253,2.738474,-6.816760e+07,43.134396,42.719009,0
4,0.587478,0.413157,61.780488,9.044897,1.502037,43.567673,41.561488,39.555302,40.759725,40.974318,1.684858,-7.888104,2.738627,-1.949416e+08,42.719009,42.355846,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,190.372500,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993,188.630005,1
1476,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,190.459545,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005,191.559998,1
1477,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,190.553182,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998,193.889999,1
1478,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,190.686818,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999,195.179993,1


In [115]:
indicators_with_price = indicators_with_price.drop(columns=['Next_Adj_Close'])
indicators_with_price

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Signal
0,0.420517,0.238064,56.052912,4.051362,0.823540,44.028288,40.539885,37.051483,40.615036,40.638546,12.020059,29.597806,2.704888,-4.208332e+08,41.546413,1
1,0.431503,0.276752,59.233517,6.192273,1.284019,44.042948,40.754083,37.465218,40.644218,40.699045,-9.379590,12.287804,2.706938,-3.257368e+08,41.999790,1
2,0.492754,0.319952,63.732478,9.481629,1.816479,43.867406,41.056252,38.245097,40.685502,40.788927,-18.948467,-5.435999,2.727886,-1.969960e+08,42.721378,1
3,0.568075,0.369577,66.042461,11.303189,1.763767,43.662505,41.356640,39.050774,40.725588,40.893170,-6.400702,-11.576253,2.738474,-6.816760e+07,43.134396,0
4,0.587478,0.413157,61.780488,9.044897,1.502037,43.567673,41.561488,39.555302,40.759725,40.974318,1.684858,-7.888104,2.738627,-1.949416e+08,42.719009,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,190.372500,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993,1
1476,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,190.459545,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005,1
1477,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,190.553182,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998,1
1478,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,190.686818,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999,1


In [116]:
y = indicators_with_price["Adj Close"]
y_2 = indicators_with_price["SMA"]
y_3 = indicators_with_price["EMA"]
y_4 = indicators_with_price["Upper_BB"]
y_5 = indicators_with_price["Middle_BB"]
y_6 = indicators_with_price["Lower_BB"]
X = np.array(date)

trace = go.Scatter(x=X, y=y, mode="lines+markers", name="Adj Close")
trace_2 = go.Scatter(x=X, y=y_2, mode="lines", name="SMA")
trace_3 = go.Scatter(x=X, y=y_3, mode="lines", name="EMA")
trace_4 = go.Scatter(x=X, y=y_4, mode="lines", name="Upper_BB")
trace_5 = go.Scatter(x=X, y=y_5, mode="lines", name="Middle_BB")
trace_6 = go.Scatter(x=X, y=y_6, mode="lines", name="Lower_BB")



layout = go.Layout(
    title='Stock Price and Volume',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Adj Close', side='left', rangemode='tozero'),
    yaxis2=dict(title='SMA', side='right', overlaying='y', rangemode='tozero'),
    yaxis3=dict(title='EMA', side='right', overlaying='y', rangemode='tozero'),
    yaxis4=dict(title='Upper_BB', side='right', overlaying='y', rangemode='tozero'),
    yaxis5=dict(title='Middle_BB', side='right', overlaying='y', rangemode='tozero'),
    yaxis6=dict(title='Lower_BB', side='right', overlaying='y', rangemode='tozero'),
    height=600,
)

fig = go.Figure(data=[trace, trace_2, trace_3, trace_4, trace_5, trace_6], layout=layout)

# Show plot
pyo.iplot(fig)

In [117]:
class RollingWindowDataset(Dataset):
    def __init__(self, X, y, window_size, device="gpu"):
        self.X = X.clone().detach().to(torch.float).to(device)
        self.y = y.clone().detach().to(torch.float).to(device)
        self.window_size = window_size

    def __len__(self):
        # Adjust the length to account for window size
        return len(self.X) - self.window_size 

    def __getitem__(self, idx):
        # Ensure idx is within the valid range
        if idx + self.window_size > len(self.X):
            raise IndexError("Index out of bounds")

        # Extract a window of data from X
        X_window = self.X[idx : idx + self.window_size]
        
        # Assuming you're predicting the value right after the window
        y_target = self.y[idx + self.window_size]  

        return X_window.clone().detach().to(torch.float), y_target.clone().detach().to(torch.float)


In [118]:
X = indicators_with_price.iloc[:, :-1]
y = indicators_with_price.iloc[:,-2]

signal_true = indicators_with_price.iloc[:,-1]
y

0        41.546413
1        41.999790
2        42.721378
3        43.134396
4        42.719009
           ...    
1475    182.679993
1476    188.630005
1477    191.559998
1478    193.889999
1479    195.179993
Name: Adj Close, Length: 1480, dtype: float64

In [119]:
train_signal_true = signal_true.iloc[:int(len(X)*0.8)]
test_signal_true = signal_true.iloc[int(len(X)*0.8):]
test_signal_true

1184    1
1185    1
1186    0
1187    1
1188    1
       ..
1475    1
1476    1
1477    1
1478    1
1479    0
Name: Signal, Length: 296, dtype: int64

In [120]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
# X_train = X_train.to_numpy()
# y_train = y_train.to_numpy()
len(y_test)

296

In [121]:
# # Get the count of each unique value in the series
# category_counts = y_train.value_counts()
# print(category_counts.index)
# print(category_counts.values)

# # Create bar plot
# plt.bar(["1","0"], category_counts.values)

# # Adding labels and title
# plt.xlabel('Category')
# plt.ylabel('Frequency')
# plt.title('Distribution of Categories')

# # Show plot
# plt.show()

In [122]:
X_train_df = pd.DataFrame()
X_test_df = pd.DataFrame()
scaler_dict = {}

for column in X_train.columns:
    # Initialize a scaler
    scaler = MinMaxScaler()
    
    # Fit on the training data and transform it
    X_train_scaled = scaler.fit_transform(X_train[column].to_numpy().reshape(-1, 1))
    X_train_df[column] = X_train_scaled.flatten()
    
    # Transform the test data using the same scaler
    X_test_scaled = scaler.transform(X_test[column].to_numpy().reshape(-1, 1))
    X_test_df[column] = X_test_scaled.flatten()

    scaler_dict[column] = scaler

    

X_train_df.head(24)

y_train_df = X_train_df["Adj Close"]
y_test_df = X_test_df["Adj Close"]
features = X_train_df.columns
X_test_df

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,0.473178,0.391409,0.484407,0.368317,0.443981,0.810260,0.788953,0.749683,0.804887,0.832500,0.850519,0.839325,0.765326,0.761787,0.780636
1,0.499847,0.411144,0.517395,0.387564,0.487570,0.813419,0.791684,0.751875,0.804933,0.833706,0.876575,0.853920,0.764804,0.775662,0.793797
2,0.523637,0.432238,0.527282,0.393264,0.448749,0.816205,0.793223,0.752021,0.804433,0.835055,0.895160,0.867699,0.724522,0.788577,0.797684
3,0.523008,0.448973,0.456308,0.359880,0.379729,0.815520,0.792793,0.751879,0.802963,0.835217,0.907286,0.887937,0.685712,0.778442,0.775317
4,0.534244,0.464868,0.497612,0.382098,0.444493,0.813808,0.792104,0.752318,0.802403,0.836117,0.913913,0.901248,0.661728,0.787383,0.790115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,0.331987,0.360467,0.219195,0.330143,0.438975,1.105702,1.104494,1.078159,1.146372,1.152418,0.696649,0.699736,0.263755,0.912203,1.018693
292,0.358007,0.354748,0.456178,0.434230,0.530971,1.099583,1.101858,1.079371,1.147028,1.152792,0.728273,0.697238,0.316961,0.925666,1.059493
293,0.396416,0.358741,0.541663,0.480565,0.534505,1.093074,1.099905,1.082407,1.147734,1.154161,0.814261,0.731129,0.316623,0.937531,1.079584
294,0.440664,0.371807,0.601115,0.515680,0.555951,1.091863,1.099564,1.083018,1.148741,1.156274,0.926300,0.813002,0.323439,0.947909,1.095561


In [123]:
scaler_dict

{'MACD': MinMaxScaler(),
 'MACD_signal': MinMaxScaler(),
 'RSI': MinMaxScaler(),
 'CMO': MinMaxScaler(),
 'MOM': MinMaxScaler(),
 'Upper_BB': MinMaxScaler(),
 'Middle_BB': MinMaxScaler(),
 'Lower_BB': MinMaxScaler(),
 'SMA': MinMaxScaler(),
 'EMA': MinMaxScaler(),
 'SLOWK': MinMaxScaler(),
 'SLOWD': MinMaxScaler(),
 'ATR': MinMaxScaler(),
 'OBV': MinMaxScaler(),
 'Adj Close': MinMaxScaler()}

In [124]:
correlation_matrix = X_train_df.corr()

# Create the heatmap with text
fig = go.Figure(data=go.Heatmap(
                    z=correlation_matrix,
                    x=correlation_matrix.columns,
                    y=correlation_matrix.columns,
                    colorscale='Viridis',
                    text=correlation_matrix.round(2).values,  # Rounded values for display
                    texttemplate="%{text}",
                    textfont={"size":9}  # Adjust text size if necessary
                    ))

# Update the layout
fig.update_layout(
    title='Correlation Matrix',
    xaxis_title="Variables",
    yaxis_title="Variables",
    xaxis=dict(side='bottom'),
    yaxis=dict(autorange='reversed'),
    width=1000,  # or any width you desire
    height=1000,  # or any height you desire
)

# Show the figure
pyo.iplot(fig)

In [125]:
X_train_tensor = torch.tensor(X_train_df.to_numpy(), dtype=torch.float, device=device)
y_train_tensor = torch.tensor(y_train_df.to_numpy(), dtype=torch.float, device=device)

X_test_tensor = torch.tensor(X_test_df.to_numpy(), dtype=torch.float, device=device)
y_test_tensor = torch.tensor(y_test_df.to_numpy(), dtype=torch.float, device=device)

train_data = RollingWindowDataset(X_train_tensor, y_train_tensor, window_size=time_step, device=device)
test_data = RollingWindowDataset(X_test_tensor, y_test_tensor, window_size=time_step, device=device)

print(test_data.__getitem__(0)[1])
print(test_data.__getitem__(1)[0])


tensor(0.7283, device='cuda:0')
tensor([[0.4998, 0.4111, 0.5174, 0.3876, 0.4876, 0.8134, 0.7917, 0.7519, 0.8049,
         0.8337, 0.8766, 0.8539, 0.7648, 0.7757, 0.7938],
        [0.5236, 0.4322, 0.5273, 0.3933, 0.4487, 0.8162, 0.7932, 0.7520, 0.8044,
         0.8351, 0.8952, 0.8677, 0.7245, 0.7886, 0.7977],
        [0.5230, 0.4490, 0.4563, 0.3599, 0.3797, 0.8155, 0.7928, 0.7519, 0.8030,
         0.8352, 0.9073, 0.8879, 0.6857, 0.7784, 0.7753],
        [0.5342, 0.4649, 0.4976, 0.3821, 0.4445, 0.8138, 0.7921, 0.7523, 0.8024,
         0.8361, 0.9139, 0.9012, 0.6617, 0.7874, 0.7901],
        [0.5474, 0.4805, 0.5145, 0.3912, 0.4671, 0.8153, 0.7928, 0.7521, 0.8022,
         0.8373, 0.9235, 0.9113, 0.6236, 0.7974, 0.7962],
        [0.5399, 0.4914, 0.4472, 0.3605, 0.4592, 0.8163, 0.7941, 0.7537, 0.8018,
         0.8374, 0.9237, 0.9172, 0.5981, 0.7914, 0.7760],
        [0.5112, 0.4936, 0.3677, 0.3216, 0.4080, 0.8078, 0.7900, 0.7546, 0.8007,
         0.8361, 0.8756, 0.9035, 0.5869, 0.7794, 0.74

# Vanilla LSTM

In [126]:
class VanillaLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob):
        super(VanillaLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size

        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = hidden_size, num_layers=layer_size,
                            dropout=(dropout_prob if layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        # Initializing hidden state
        h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Initialize cell state
        c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

# Stacked LSTM

In [127]:
class StackedLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob, stateful):
        super(StackedLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size
        self.stateful = stateful

        self.hn = None
        self.cn = None

        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = hidden_size, num_layers=layer_size,
                            dropout=(dropout_prob if layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def reset_hidden_state(self):
        self.hn = None
        self.cn = None

    def forward(self, x):
        # If the model is not stateful or it's the first batch (states are None), initialize h0 and c0
        if not self.stateful or self.hn is None:
            h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(x.device)
            c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(x.device)
        else:
            # If the model is stateful and it's not the first batch, use the stored states
            h0, c0 = self.hn, self.cn

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        if self.stateful:
            self.hn, self.cn = hn.detach(), cn.detach()

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

# Bidirectional LSTM

# 1D CNN-LSTM

In [128]:
class OneDimCNNLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob, conv_channels, kernel_size, pool_size, stride):
        super(OneDimCNNLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size

        # Convolutional Layer
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=conv_channels, kernel_size=kernel_size)
        self.relu1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_size)

        self.conv2 = nn.Conv1d(in_channels=conv_channels, out_channels=conv_channels*2, kernel_size=kernel_size)
        self.relu2 = nn.ReLU()
        self.maxpool2 = nn.MaxPool1d(kernel_size=2, stride=2)


        self.lstm = nn.LSTM(input_size = conv_channels*2, hidden_size = hidden_size, num_layers=layer_size,
                            dropout=(dropout_prob if layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        # CNN in_dim: (batch_size, in_channels, length)
        x = x.transpose(1, 2)

        # Conv layer forward propagate
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.maxpool2(x)

        # LSTM in_dim: (batch_size, seq_len, input_size)
        x = x.transpose(1, 2)

        assert x.size(-1) == self.lstm.input_size, f"Mismatch in LSTM input size. Expected: {self.lstm.input_size}, Got: {x.size(-1)}"

        # Initializing hidden state
        h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Initialize cell state
        c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

In [129]:
class ModelActioner:
    
    def __init__(self, train_data, test_data, device, model_type):
        self.train_data = train_data
        self.test_data = test_data
        self.device = device
        self.model_type = model_type
        self.model = None
        self.optimizer = None
        self.criterion = nn.MSELoss()

    
    def train_validate(self, config, trial):

        if self.model_type == "Vanilla LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = 1
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]

            self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=1).to(self.device)
            
        elif self.model_type == "Stacked LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            
            self.model = StackedLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=1).to(self.device)

        elif self.model_type == "1D CNN-LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            kernel_size = config["kernel_size"]
            conv_channels = config["conv_channels"]
            pool_size = config["pool_size"]
            stride = config["stride"]

            self.model = OneDimCNNLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=1, conv_channels=conv_channels, kernel_size=kernel_size, pool_size=pool_size, stride=stride).to(self.device)

        n_splits = 5
        tscv = TimeSeriesSplit(n_splits=n_splits)

        val_losses = []

        for fold, (train_ids, val_ids) in enumerate(tscv.split(self.train_data)):
            print(f'Starting fold {fold+1}:')

            self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

            scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True) 

            train_subset = Subset(self.train_data, train_ids)
            val_subset = Subset(self.train_data, val_ids)
            
            # Creating data loader
            train_loader = DataLoader(dataset=train_subset, batch_size=batch_size, shuffle=False)
            val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False)

            # Training Loop
            for epoch in range(epochs):
                print('epochs {}/{}'.format(epoch+1,epochs))

                running_loss = 0.0
                total_sample_train = 0

                self.model.train()

                for batch_idx, (data, target) in enumerate(train_loader):

                    data, target = data.to(self.device), target.to(self.device)
                    target = target.view(-1, 1)  # Reshape target to have an extra dimension


                    self.optimizer.zero_grad()
                    preds = self.model(data)

                    loss = self.criterion(preds, target)
                    loss.backward()
                    self.optimizer.step() # Update model params

                    running_loss += loss.item() * data.size(0)
                    total_sample_train += data.size(0)

                train_loss = running_loss/total_sample_train
                #print(f"train loss: {train_loss}")

                self.model.eval()
                val_running_loss = 0.0
                total_sample_val = 0

                with torch.no_grad():

                    for batch_idx, (data, target) in enumerate(val_loader):
                        data, target = data.to(self.device), target.to(self.device)
                        target = target.view(-1, 1)

                        preds = self.model(data)
                        loss = self.criterion(preds, target)

                        val_running_loss += loss.item() * data.size(0)
                        total_sample_val += data.size(0)
                
                if trial.should_prune():
                    raise optuna.TrialPruned()
                
                val_loss = val_running_loss/total_sample_val
                val_losses.append(val_loss)
                scheduler.step(val_loss)
                
                unique_step = fold * epochs + epoch
                trial.report(val_loss, unique_step)

                current_lr = self.optimizer.param_groups[0]['lr']

                print(f'Current Learning Rate: {current_lr}')
                print(f"train_loss: {train_loss}, val_loss: {val_loss}")
                
        mean_val_loss = np.mean(val_losses)
        print(f"Mean validation loss: {mean_val_loss}")
        return mean_val_loss
    
                    
    def train(self, config):
        if self.model_type == "Vanilla LSTM":

            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = 1
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]

            self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=1).to(self.device)
            
        elif self.model_type == "Stacked LSTM":
            
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            
            self.model = StackedLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=1).to(self.device)

        elif self.model_type == "1D CNN-LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            kernel_size = config["kernel_size"]
            conv_channels = config["conv_channels"]
            pool_size = config["pool_size"]
            stride = config["stride"]

            self.model = OneDimCNNLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=1, conv_channels=conv_channels, kernel_size=kernel_size, pool_size=pool_size, stride=stride).to(self.device)

        # Update optimizer with updated lr
        self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

        # Creating data loader
        train_loader = DataLoader(dataset=self.train_data, batch_size=batch_size, shuffle=False)

        scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True)  

        # Training Loop
        for epoch in range(epochs):
            print('epochs {}/{}'.format(epoch+1,epochs))

            running_loss = 0.0
            total_sample_train = 0

            self.model.train()

            for batch_idx, (data, target) in enumerate(train_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.view(-1, 1)  # Reshape target to have an extra dimension

                self.optimizer.zero_grad()
                preds = self.model(data)

                loss = self.criterion(preds, target)
                loss.backward()
                self.optimizer.step() # Update model params

                running_loss += loss.item() * data.size(0)
                total_sample_train += data.size(0)

            train_loss = running_loss/total_sample_train
            #print(f"train loss: {train_loss}")
            scheduler.step(train_loss)
            current_lr = self.optimizer.param_groups[0]['lr']

            print(f'Current Learning Rate: {current_lr}')
            print(f"train_loss: {train_loss}")
        
        return self.model
            
    
    def test(self, config):
        batch_size = config["batch_size"]
        all_preds = []

        test_loader = DataLoader(dataset=self.test_data, batch_size=batch_size, shuffle=False)

        running_loss = .0
        total_sample = 0

        self.model.eval()

        with torch.no_grad():

            for batch_idx, (data, target) in enumerate(test_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.view(-1, 1)
                
                preds = self.model(data)
                loss = self.criterion(preds, target)

                running_loss += loss.item() * data.size(0)
                total_sample += data.size(0)

                all_preds.extend(preds.cpu().numpy())

            test_loss = running_loss/total_sample
            print(f"test_loss: {test_loss}")

        return all_preds
    


In [130]:
model_type = "Stacked LSTM"

def objective(trial):
    if model_type == "Vanilla LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.15),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 10, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9)
        }

    elif model_type == "Stacked LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.2),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 5, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9),
            "num_layers": trial.suggest_int("num_layers", 1, 5)
        }

    elif model_type == "1D CNN-LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.2),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 5, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9),
            "conv_channels": trial.suggest_int("conv_channels", 16, 128),
            "kernel_size": trial.suggest_int("kernel_size", 2, 6),
            "num_layers": trial.suggest_int("num_layers", 1, 5),
            "pool_size": trial.suggest_int("pool_size", 2, 5),
            "stride": trial.suggest_int("stride", 1, 4)
        }

    trainer = ModelActioner(train_data, test_data, device, model_type)

    val_loss = trainer.train_validate(config, trial)

    return val_loss

In [131]:
study_name = "Vanilla-LSTM-Tunner"
storage_url = "sqlite:///db.sqlite3"

optuna.delete_study(study_name=study_name, storage= storage_url)

study = optuna.create_study(direction='minimize', 
                            storage=storage_url, 
                            sampler=TPESampler(),
                            pruner=optuna.pruners.HyperbandPruner(
                            min_resource=30,  
                            max_resource=150,  
                            reduction_factor=3,  
                            ),
                            study_name=study_name,
                            load_if_exists=False)

pbar = tqdm(total=40, desc='Optimizing', unit='trial')

def callback(study, trial):
    # Update the progress bar
    pbar.update(1)
    # Optionally, you can include additional information in the progress bar
    pbar.set_postfix_str(f"Best Value: {study.best_value:.4f}")


study.optimize(objective, n_trials=40, callbacks=[callback])
pbar.close()

# Best hyperparameters
print('Number of finished trials:', len(study.trials))
print('Best trial:')
trial = study.best_trial

print('Value:', trial.value)
print('Params:')
for key, value in trial.params.items():
    print(f'{key}: {value}')

[I 2024-01-26 13:12:03,229] A new study created in RDB with name: Vanilla-LSTM-Tunner


Optimizing:   0%|          | 0/40 [00:00<?, ?trial/s]

Starting fold 1:
epochs 1/116
Current Learning Rate: 1.579787598700664e-05
train_loss: 0.015491704922169447, val_loss: 0.021475601941347122
epochs 2/116
Current Learning Rate: 1.579787598700664e-05
train_loss: 0.015188372693955898, val_loss: 0.02109705563634634
epochs 3/116
Current Learning Rate: 1.579787598700664e-05
train_loss: 0.015601668041199446, val_loss: 0.02072245953604579
epochs 4/116
Current Learning Rate: 1.579787598700664e-05
train_loss: 0.014754651580005884, val_loss: 0.02035200409591198
epochs 5/116
Current Learning Rate: 1.579787598700664e-05
train_loss: 0.014167014043778181, val_loss: 0.019986337050795555
epochs 6/116
Current Learning Rate: 1.579787598700664e-05
train_loss: 0.013959396630525589, val_loss: 0.019624963402748108
epochs 7/116
Current Learning Rate: 1.579787598700664e-05
train_loss: 0.013510598801076412, val_loss: 0.019268137402832508
epochs 8/116
Current Learning Rate: 1.579787598700664e-05
train_loss: 0.013261628337204456, val_loss: 0.018915643449872732
ep

[I 2024-01-26 13:13:00,026] Trial 0 finished with value: 0.026604424174478607 and parameters: {'batch_size': 95, 'epochs': 116, 'hidden_size': 91, 'learning_rate': 1.579787598700664e-05, 'dropout_prob': 0.08742803388397978, 'weight_decay': 4.62898458132716e-06, 'lr_step_size': 29, 'gamma': 0.5327445559368696, 'num_layers': 1}. Best is trial 0 with value: 0.026604424174478607.


Current Learning Rate: 1.579787598700664e-05
train_loss: 0.0011590053647523746, val_loss: 0.0015053260140120983
Mean validation loss: 0.026604424174478607
Starting fold 1:
epochs 1/148
Current Learning Rate: 4.877021798443148e-05
train_loss: 0.008939117036367718, val_loss: 0.012349283283478335
epochs 2/148
Current Learning Rate: 4.877021798443148e-05
train_loss: 0.008125652600766012, val_loss: 0.011657222242731797
epochs 3/148
Current Learning Rate: 4.877021798443148e-05
train_loss: 0.007814860565122216, val_loss: 0.011012230617435354
epochs 4/148
Current Learning Rate: 4.877021798443148e-05
train_loss: 0.006982155866602338, val_loss: 0.010410580235092264
epochs 5/148
Current Learning Rate: 4.877021798443148e-05
train_loss: 0.00691282464816284, val_loss: 0.009845203984724847
epochs 6/148
Current Learning Rate: 4.877021798443148e-05
train_loss: 0.006275974516756833, val_loss: 0.009314828836604169
epochs 7/148
Current Learning Rate: 4.877021798443148e-05
train_loss: 0.0062062069448936535

...

Current Learning Rate: 0.00016980864741844437
train_loss: 0.001086160689929353, val_loss: 0.0016730188493200235
epochs 92/117
Starting fold 1:
epochs 1/99
Current Learning Rate: 1.283341718268883e-06
train_loss: 0.0024249984539653126, val_loss: 0.0027643673054530823
epochs 2/99
Current Learning Rate: 1.283341718268883e-06
train_loss: 0.002040190965329346, val_loss: 0.0027631790006508758
epochs 3/99
Current Learning Rate: 1.283341718268883e-06
train_loss: 0.0021916057847097123, val_loss: 0.00276262892906456
epochs 4/99
Current Learning Rate: 1.283341718268883e-06
train_loss: 0.0019851628917661544, val_loss: 0.0027620590821913395
epochs 5/99
Current Learning Rate: 1.283341718268883e-06
train_loss: 0.0020803430693616207, val_loss: 0.0027615004648013333
epochs 6/99
Current Learning Rate: 1.283341718268883e-06
train_loss: 0.002073521480748528, val_loss: 0.0027609759479099395
epochs 7/99
Current Learning Rate: 1.283341718268883e-06
train_loss: 0.002013303308845743, val_loss: 0.00276041502978

[I 2024-01-26 13:26:36,000] Trial 38 pruned. 


Current Learning Rate: 1.283341718268883e-06
train_loss: 0.0020355177540822248, val_loss: 0.002742416640486274
epochs 32/99
Starting fold 1:
epochs 1/146
Current Learning Rate: 0.014620234838549195
train_loss: 0.09345318987279345, val_loss: 0.0021580456033054936
epochs 2/146
Current Learning Rate: 0.014620234838549195
train_loss: 0.0016132292803376913, val_loss: 0.0012191601073075283
epochs 3/146
Current Learning Rate: 0.014620234838549195
train_loss: 0.0012836639193425838, val_loss: 0.002096406689624449
epochs 4/146
Current Learning Rate: 0.014620234838549195
train_loss: 0.0011944876505846256, val_loss: 0.0007683285512030125
epochs 5/146
Current Learning Rate: 0.014620234838549195
train_loss: 0.0005393405444920063, val_loss: 0.004452039938003413
epochs 6/146
Current Learning Rate: 0.014620234838549195
train_loss: 0.011486023076270757, val_loss: 0.01354057488864974
epochs 7/146
Current Learning Rate: 0.014620234838549195
train_loss: 0.007576091067963525, val_loss: 0.002954696170299461


[I 2024-01-26 13:26:59,082] Trial 39 pruned. 


Number of finished trials: 40
Best trial:
Value: 0.0011512266347036932
Params:
batch_size: 126
epochs: 108
hidden_size: 55
learning_rate: 0.0011771377657156651
dropout_prob: 0.127914833127579
weight_decay: 1.1312951163760579e-05
lr_step_size: 36
gamma: 0.6414565779787799
num_layers: 1


In [132]:
trainer = ModelActioner(train_data, test_data, device, model_type)
model = trainer.train(trial.params)

epochs 1/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.22239974339149501
epochs 2/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.039227310180860134
epochs 3/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.07507393879962987
epochs 4/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.045202181346126295
epochs 5/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.023447028561005074
epochs 6/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.016177093433706383
epochs 7/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.009941333281121364
epochs 8/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.006076831294615802
epochs 9/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.009177255476702397
epochs 10/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.007298129611942721
epochs 11/108
Current Learning Rate: 0.0011771377657156651
train_loss: 0.01469885814983986


In [133]:
y_true = y_test_df[time_step:]
preds = trainer.test(trial.params)

# Mean Absolute Error
mae = mean_absolute_error(y_true, preds)
print(f'Mean Absolute Error: {mae}')

# Mean Squared Error
mse = mean_squared_error(y_true, preds)
print(f'Mean Squared Error: {mse}')

# Root Mean Squared Error
rmse = np.sqrt(mse)
print(f'Root Mean Squared Error: {rmse}')

# R-squared Score
r2 = r2_score(y_true, preds)
print(f'R-squared Score: {r2}')

mape = mean_absolute_percentage_error(y_true, preds)*100
print(f"MAPE: {mape}")

test_loss: 0.001196102937683463
Mean Absolute Error: 0.029745363273016752
Mean Squared Error: 0.001196102764509944
Root Mean Squared Error: 0.03458471865593161
R-squared Score: 0.8839130177836735
MAPE: 3.011679615510095


In [134]:
y_true

44     0.728267
45     0.737950
46     0.733381
47     0.747701
48     0.761134
         ...   
291    1.018693
292    1.059493
293    1.079584
294    1.095561
295    1.104407
Name: Adj Close, Length: 252, dtype: float64

In [162]:
y_true = y_true.reset_index(drop=True)
#y_true = y_true.drop(columns=["index","level_0"])
X_value = X_test_df.iloc[43:,-1]
X_value = X_value.reset_index()
preds_values = [x[0] for x in preds]
X_value["Preds"] = pd.Series(preds_values)
X_value["Next Day"] = y_true
X_value = X_value.drop(columns=["index"])
X_value["Diff Preds"] = (X_value["Preds"] - X_value["Adj Close"])
X_value["Diff Next Day"] = (X_value["Next Day"] - X_value["Adj Close"])
X_value = X_value.dropna()
X_value

,Adj Close,Preds,Next Day,Diff Preds,Diff Next Day
0,0.706173,0.693062,0.728267,-0.013111,0.022093
1,0.728267,0.705377,0.737950,-0.022890,0.009683
2,0.737950,0.715592,0.733381,-0.022357,-0.004569
3,0.733381,0.719320,0.747701,-0.014061,0.014320
4,0.747701,0.726739,0.761134,-0.020962,0.013433
...,...,...,...,...,...
247,1.025208,0.993976,1.018693,-0.031231,-0.006514
248,1.018693,0.985355,1.059493,-0.033338,0.040800
249,1.059493,1.001208,1.079584,-0.058285,0.020091
250,1.079584,1.018027,1.095561,-0.061557,0.015977


In [163]:
def assign_label(diff):
    if diff > 0.001:
        return 1  # Buy
    elif diff < -0.05:
        return 0  # Sell
    else:
        return 2  # Hold/Other

In [164]:
# Apply the function to assign labels
X_value['Signal Preds'] = X_value['Diff Preds'].apply(assign_label)
X_value['Signal True'] = X_value['Diff Next Day'].apply(assign_label)

# Dropping the intermediate difference columns if they are no longer needed
X_value = X_value.drop(columns=['Diff Preds', 'Diff Next Day'])

X_value

,Adj Close,Preds,Next Day,Signal Preds,Signal True
0,0.706173,0.693062,0.728267,2,1
1,0.728267,0.705377,0.737950,2,1
2,0.737950,0.715592,0.733381,2,2
3,0.733381,0.719320,0.747701,2,1
4,0.747701,0.726739,0.761134,2,1
...,...,...,...,...,...
247,1.025208,0.993976,1.018693,2,2
248,1.018693,0.985355,1.059493,2,1
249,1.059493,1.001208,1.079584,0,1
250,1.079584,1.018027,1.095561,0,1


In [165]:
accuracy = accuracy_score(X_value["Signal True"], X_value["Signal Preds"])
print(f'Accuracy: {accuracy * 100:.2f}%')

# Precision
precision = precision_score(X_value["Signal True"], X_value["Signal Preds"], average='weighted')
print(f'Precision: {precision:.4f}')

# Recall
recall = recall_score(X_value["Signal True"], X_value["Signal Preds"], average='weighted')
print(f'Recall: {recall:.4f}')

# F1 Score
f1 = f1_score(X_value["Signal True"], X_value["Signal Preds"], average='weighted')
print(f'F1 Score: {f1:.4f}')


Accuracy: 47.62%
Precision: 0.6666
Recall: 0.4762
F1 Score: 0.3439


In [166]:
y_true

0      0.728267
1      0.737950
2      0.733381
3      0.747701
4      0.761134
         ...   
247    1.018693
248    1.059493
249    1.079584
250    1.095561
251    1.104407
Name: Adj Close, Length: 252, dtype: float64

In [167]:
X_value["Signal Preds"]

0      2
1      2
2      2
3      2
4      2
      ..
247    2
248    2
249    0
250    0
251    0
Name: Signal Preds, Length: 252, dtype: int64

In [168]:
signals = pd.DataFrame(X_value["Signal Preds"].values, columns=['Signal'])
signals

,Signal
0,2
1,2
2,2
3,2
4,2
...,...
247,2
248,2
249,0
250,0


In [169]:
signals["Signal"].value_counts()

Signal
2    236
0     10
1      6
Name: count, dtype: int64

In [170]:
signals["Date"] = date_test
signals

,Signal,Date
0,2,2023-01-23
1,2,2023-01-24
2,2,2023-01-25
3,2,2023-01-26
4,2,2023-01-27
...,...,...
247,2,2024-01-17
248,2,2024-01-18
249,0,2024-01-19
250,0,2024-01-22


In [171]:
stock_price = stock_data["Adj Close"].iloc[test_index:]
stock_price=stock_price.reset_index()
stock_price=stock_price.drop(columns=["index"])
stock_price

,Adj Close
0,140.325653
1,141.737762
2,141.071487
3,143.159821
4,145.118835
...,...
247,182.679993
248,188.630005
249,191.559998
250,193.889999


In [172]:
date_test["Date"] = date_test["Date"].dt.strftime('%Y-%m-%d')
date_test

AttributeError: Can only use .dt accessor with datetimelike values

In [173]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(date_test).flatten(), y=stock_data["Adj Close"].iloc[test_index:], mode='lines', name='TSLA Stock Price'))

# Buy and sell signals
buy_signals = signals[signals['Signal'] == 1]
sell_signals = signals[signals['Signal'] == 0]

# Ensure that buy and sell prices are aligned with the signals by matching on the 'Date' column
buy_prices = stock_data[stock_data['Date'].isin(buy_signals['Date'])]["Adj Close"]
sell_prices = stock_data[stock_data['Date'].isin(sell_signals['Date'])]["Adj Close"]

# Plot buy signals
fig.add_trace(go.Scatter(
    x=buy_signals['Date'], 
    y=buy_prices, 
    mode='markers', 
    name='Buy', 
    marker=dict(color='green', size=10, symbol='triangle-up')
))

# Plot sell signals
fig.add_trace(go.Scatter(
    x=sell_signals['Date'], 
    y=sell_prices, 
    mode='markers', 
    name='Sell', 
    marker=dict(color='red', size=10, symbol='triangle-down')
))

# Update layout
fig.update_layout(
    title='Stock Price with Buy and Sell Signals',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    height = 700,
    width=1280
)

# Show the plot
pyo.iplot(fig)

In [174]:
price_signal = stock_data[stock_data['Date'].isin(signals['Date'])]["Adj Close"]
price_signal = price_signal.reset_index()
price_signal = price_signal.drop(columns=["index"])
result = pd.concat([price_signal,signals], axis=1)
result

,Adj Close,Signal,Date
0,140.325653,2,2023-01-23
1,141.737762,2,2023-01-24
2,141.071487,2,2023-01-25
3,143.159821,2,2023-01-26
4,145.118835,2,2023-01-27
...,...,...,...
247,182.679993,2,2024-01-17
248,188.630005,2,2024-01-18
249,191.559998,0,2024-01-19
250,193.889999,0,2024-01-22


In [175]:
price_signal

,Adj Close
0,140.325653
1,141.737762
2,141.071487
3,143.159821
4,145.118835
...,...
247,182.679993
248,188.630005
249,191.559998
250,193.889999


In [149]:
sell_signals

,Signal,Date
148,0,2023-08-24
152,0,2023-08-30
153,0,2023-08-31
154,0,2023-09-01
155,0,2023-09-05
156,0,2023-09-06
210,0,2023-11-21
249,0,2024-01-19
250,0,2024-01-22
251,0,2024-01-23


In [150]:
buy_prices

Series([], Name: Adj Close, dtype: float64)